# 📊 SCAF – Semantic-Contextual Adaptive Forecasting
## Prévision financière avec détection de régimes et agrégation dynamique

Ce notebook implémente le framework **SCAF** (Semantic-Contextual Adaptive Forecasting), qui combine des modèles spécialisés (ARIMA, GARCH, LSTM) avec une sélection dynamique basée sur le régime de marché détecté via clustering et contexte textuel simulé.

In [ ]:
# %%capture
!pip install yfinance pandas numpy matplotlib statsmodels arch tensorflow scikit-learn fpdf

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from statsmodels.tsa.arima.model import ARIMA
from arch import arch_model
from keras.models import Sequential
from keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, confusion_matrix, ConfusionMatrixDisplay
from fpdf import FPDF

## 🔹 1. Téléchargement des données financières

In [ ]:
def fetch_real_data(ticker="AAPL", period="2y", interval="1d"):
    """Download market data from Yahoo Finance with synthetic fallback."""
    try:
        data = yf.download(ticker, period=period, interval=interval, progress=False)
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)
        data = data[['Close']].dropna()
        if len(data) >= 200:
            return data.reset_index()
    except Exception:
        pass
    print("[INFO] Données réelles indisponibles – génération de données synthétiques AAPL-like.")
    return _generate_synthetic_data()

def _generate_synthetic_data(n_days=504, seed=42):
    """Generate realistic synthetic price series (2 years of daily data)."""
    rng = np.random.default_rng(seed)
    # GBM with regime-switching volatility
    dates = pd.date_range(end='2024-12-31', periods=n_days, freq='B')
    price = 150.0
    prices = [price]
    vol_regime = 0.01  # start with low-vol
    for i in range(1, n_days):
        # Regime switch probability
        if rng.random() < 0.01:
            vol_regime = rng.choice([0.008, 0.018, 0.025])
        ret = rng.normal(0.0003, vol_regime)
        price = price * (1 + ret)
        prices.append(price)
    df_synth = pd.DataFrame({'Date': dates, 'Close': prices})
    return df_synth

df = fetch_real_data("AAPL", period="2y", interval="1d")
df = df[['Date', 'Close']].rename(columns={'Close': 'value', 'Date': 'timestamp'})
df['timestamp'] = pd.to_datetime(df['timestamp']).dt.to_period('D')
print(f"Données chargées : {len(df)} jours, prix min={df['value'].min():.2f}, max={df['value'].max():.2f}")

## 🔹 2. Génération de contextes textuels simulés

In [ ]:
def generate_contextual_descriptions(data_col):
    context = []
    data = data_col.values
    for i in range(len(data)):
        if i == 0:
            change = 0
        else:
            change = (data[i] - data[i-1]) / data[i-1] * 100

        if change > 1.0:
            ctx = "Market is experiencing a strong positive movement."
        elif change < -1.0:
            ctx = "Market is experiencing a strong negative movement."
        else:
            ctx = "Market is relatively stable today."

        context.append(ctx)
    return context

df['context'] = generate_contextual_descriptions(df['value'])

## 🔹 3. Détection des régimes (non-stationnarité)

In [ ]:
def detect_regimes(data, n_clusters=3):
    """Cluster market regimes by return magnitude; assign labels by volatility level."""
    rets = np.log(data / data.shift(1)).dropna()
    returns_2d = rets.values.reshape(-1, 1)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(returns_2d)
    # Map cluster IDs to regime names based on intra-cluster volatility
    cluster_vols = {}
    for cid in range(n_clusters):
        mask = labels == cid
        cluster_vols[cid] = returns_2d[mask].std() if mask.sum() > 1 else 0
    sorted_clusters = sorted(cluster_vols, key=cluster_vols.get)
    vol_map = {sorted_clusters[0]: 0,   # lowest vol  -> 0 = low volatility
               sorted_clusters[1]: 2,   # mid vol     -> 2 = trending
               sorted_clusters[2]: 1}   # highest vol -> 1 = high volatility
    remapped = np.array([vol_map[l] for l in labels])
    regimes = pd.Series(remapped, index=data.index[1:], dtype=int)
    regimes = regimes.reindex(data.index, fill_value=0)
    return regimes

df['regime'] = detect_regimes(df['value'])
df['regime_label'] = df['regime'].map({0: "low volatility", 1: "high volatility", 2: "trending"})
print("Distribution des régimes :")
print(df['regime_label'].value_counts())


## 🔹 4. Définition des modèles spécialisés

In [ ]:
scaler = MinMaxScaler()

def _returns(data):
    """Compute percentage returns from price series."""
    return data.pct_change().dropna() * 100  # arch expects %-scale

def train_arima(data):
    model = ARIMA(data, order=(1, 0, 0))
    return model.fit()

def train_garch(data):
    """Fit GARCH on daily returns (%-scale) for conditional variance modelling."""
    rets = _returns(data)
    model = arch_model(rets, vol='GARCH', p=1, o=0, q=1, mean='AR', lags=1)
    return model.fit(disp='off')

def train_lstm(data, seq_length=10):
    data_scaled = scaler.fit_transform(np.array(data).reshape(-1, 1))
    X, y = [], []
    for i in range(seq_length, len(data_scaled)):
        X.append(data_scaled[i-seq_length:i])
        y.append(data_scaled[i])
    X, y = np.array(X), np.array(y)
    model = Sequential([
        LSTM(10, input_shape=(X.shape[1], 1)),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    model.fit(X, y, epochs=10, batch_size=32, verbose=0)
    return model, scaler


## 🔹 5. Définition de la classe SCAF

In [ ]:
class SCAF:
    def __init__(self):
        self.models = {}
        self.weights = {}
        self.scaler = MinMaxScaler()

    def fit(self, data, regime):
        if regime == "low volatility":
            self.models["ARIMA"] = train_arima(data)
            self.weights["ARIMA"] = 1.0
        elif regime == "high volatility":
            self.models["GARCH"] = train_garch(data)
            self.weights["GARCH"] = 1.0
        elif regime == "trending":
            self.models["LSTM"], self.scaler = train_lstm(data)
            self.weights["LSTM"] = 1.0
        else:
            self.models["ARIMA"] = train_arima(data)
            self.models["GARCH"] = train_garch(data)
            self.models["LSTM"], self.scaler = train_lstm(data)
            self.weights = {"ARIMA": 0.4, "GARCH": 0.2, "LSTM": 0.4}

    def predict(self, data):
        predictions = {}
        last_price = float(data.iloc[-1])
        for name, model in self.models.items():
            if name == "ARIMA":
                # ARIMA forecasts next price level directly
                predictions[name] = float(model.forecast(steps=1).iloc[0])
            elif name == "GARCH":
                # GARCH is fitted on returns (%-scale); mean forecast -> next price
                res = model.forecast(horizon=1, reindex=False)
                mean_return_pct = float(res.mean.iloc[-1, 0])
                predictions[name] = last_price * (1.0 + mean_return_pct / 100.0)
            elif name == "LSTM":
                data_scaled = self.scaler.transform(np.array(data).reshape(-1, 1))
                pred = model.predict(data_scaled[-10:].reshape(1, 10, 1), verbose=0)
                predictions[name] = float(self.scaler.inverse_transform(pred)[0, 0])
        final_pred = sum(self.weights[name] * pred for name, pred in predictions.items())
        return final_pred, predictions


## 🔹 6. Exécution de SCAF et évaluation

In [ ]:
train_size = int(len(df) * 0.8)
train_data = df.iloc[:train_size]
test_data = df.iloc[train_size:]

scaf = SCAF()
scaf.fit(train_data['value'], df['regime_label'][train_size - 1])

true_values = test_data['value']
pred_values = []
pred_details = []

for i in range(len(test_data)):
    idx = train_size + i
    pred, details = scaf.predict(df['value'][:idx])
    pred_values.append(pred)
    pred_details.append(details)

test_data['pred'] = pred_values
test_data['error'] = abs(test_data['value'] - test_data['pred'])

rmse = np.sqrt(mean_squared_error(true_values, pred_values))
mae = mean_absolute_error(true_values, pred_values)

## 🔹 7. Génération du rapport de performance

In [ ]:
def generate_performance_report(test_data, pred_details, rmse, mae):
    print("📊 RAPPORT DE PERFORMANCE – SCAF")
    print("==================================")
    print(f"RMSE : {rmse:.4f}")
    print(f"MAE  : {mae:.4f}")
    print("\n")

    # Use .get() to avoid KeyError when a model is not active for a given regime
    test_data = test_data.copy()
    for model_name in ["ARIMA", "GARCH", "LSTM"]:
        col = f"error_{model_name}"
        errors = [abs(v - p.get(model_name, float('nan')))
                  for v, p in zip(test_data['value'], pred_details)]
        test_data[col] = errors

    error_cols = [c for c in ['error_ARIMA', 'error_GARCH', 'error_LSTM']
                  if test_data[c].notna().any()]
    regime_errors = test_data.groupby('regime_label')[error_cols].mean()
    print("📈 ERREUR MOYENNE PAR RÉGIME")
    print(regime_errors.round(4))

    plt.figure(figsize=(12, 6))
    plt.plot(test_data['timestamp'].astype(str), test_data['value'], label='Valeurs réelles')
    plt.plot(test_data['timestamp'].astype(str), test_data['pred'], label='Prédictions SCAF', color='red')
    plt.fill_between(test_data['timestamp'].astype(str),
                     test_data['pred'] - test_data['error'],
                     test_data['pred'] + test_data['error'],
                     color='orange', alpha=0.2, label='Incertitude')
    plt.title("Prédictions SCAF avec incertitude")
    plt.xlabel("Date")
    plt.ylabel("Prix")
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('/tmp/scaf_predictions.png', dpi=100)
    plt.show()
    print("[Graphique sauvegardé dans /tmp/scaf_predictions.png]")

    # Regime prediction based on active model
    regime_map = {"ARIMA": "low volatility", "GARCH": "high volatility", "LSTM": "trending"}
    predicted_regimes = []
    for p in pred_details:
        if p:
            best_model = max(p, key=lambda k: abs(p[k] - 0))
            predicted_regimes.append(regime_map.get(best_model, "low volatility"))
        else:
            predicted_regimes.append("low volatility")

    cm = confusion_matrix(test_data['regime_label'], predicted_regimes,
                          labels=["low volatility", "high volatility", "trending"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=["low", "high", "trending"])
    disp.plot(cmap=plt.cm.Blues)
    plt.title("Matrice de confusion : Régime réel vs Régime prédit")
    plt.savefig('/tmp/scaf_confusion_matrix.png', dpi=100)
    plt.show()
    print("[Matrice de confusion sauvegardée dans /tmp/scaf_confusion_matrix.png]")

generate_performance_report(test_data, pred_details, rmse, mae)